# 05 · Momentum and Non-Convex Optimization Landscapes

While vanilla Gradient Descent works well on isotropic convex surfaces, machine learning loss landscapes frequently exhibit:
1. **Ill-conditioned ravines:** Regions where the surface curves much more steeply in one dimension than another (high Hessian condition number).
2. **Saddle points:** Stationary points where $\nabla f = 0$ but the Hessian possesses both positive and negative eigenvalues.
3. **Local minima and plateaus:** Regions where vanilla gradient descent stalls due to vanishing gradients.

**Polyak's Momentum** (Polyak, 1964) solves these pathologies by modeling a heavy physical particle rolling down a potential well:

$$v_t = \gamma v_{t-1} + \eta \nabla f(\theta_{t-1})$$
$$\theta_t = \theta_{t-1} - v_t$$

In this notebook, we explore:
1. Mathematical analysis of momentum as an exponentially weighted moving average of past gradients.
2. Navigating an ill-conditioned anisotropic quadratic ravine: Vanilla GD vs. Momentum GD.
3. Escaping saddle points and non-convex potential wells.
4. Convergence trajectories and loss curves.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# Anisotropic Quadratic Ravine: f(x, y) = 0.05 * x^2 + 1.0 * y^2
# Curvature in y is 20x steeper than in x (Condition number kappa = 20)
def ravine_loss(x: float, y: float) -> float:
    return 0.05 * x * x + 1.0 * y * y


def ravine_grad(x: float, y: float) -> tuple[float, float]:
    return 0.10 * x, 2.0 * y

## 1. Mathematical Mechanics of Momentum

Unrolling the velocity recurrence:

$$v_t = \eta \sum_{\tau=0}^{t-1} \gamma^\tau \nabla f(\theta_{t-1-\tau})$$

For directions where gradients consistently point the same way (along the flat floor of the valley), the terms constructively accumulate:

$$v_\parallel \approx \frac{\eta}{1 - \gamma} \nabla f_\parallel$$

With $\gamma = 0.9$, the effective step size accelerates by a factor of $\frac{1}{1 - 0.9} = 10\times$!

Conversely, in directions where gradients oscillate back and forth (between the steep walls of the ravine), consecutive terms have opposing signs and cancel out, damping cross-ravine oscillations.


In [ ]:
def optimize_vanilla_gd(init_pt, lr, steps=50):
    x, y = init_pt
    traj = [(x, y)]
    for _ in range(steps):
        gx, gy = ravine_grad(x, y)
        x -= lr * gx
        y -= lr * gy
        traj.append((x, y))
    return np.array(traj)


def optimize_momentum_gd(init_pt, lr, gamma=0.9, steps=50):
    x, y = init_pt
    vx, vy = 0.0, 0.0
    traj = [(x, y)]
    for _ in range(steps):
        gx, gy = ravine_grad(x, y)
        vx = gamma * vx + lr * gx
        vy = gamma * vy + lr * gy
        x -= vx
        y -= vy
        traj.append((x, y))
    return np.array(traj)


start_pt = (10.0, 3.0)
lr = 0.45
traj_vanilla = optimize_vanilla_gd(start_pt, lr=lr, steps=60)
traj_momentum = optimize_momentum_gd(start_pt, lr=lr, gamma=0.85, steps=60)

print(
    f"Vanilla GD final position after 60 steps:  ({traj_vanilla[-1, 0]:.4f}, {traj_vanilla[-1, 1]:.4f})"
)
print(
    f"Momentum GD final position after 60 steps: ({traj_momentum[-1, 0]:.4f}, {traj_momentum[-1, 1]:.4f})"
)
assert abs(traj_momentum[-1, 0]) < abs(traj_vanilla[-1, 0])

In [ ]:
# Plotting trajectories on anisotropic contour
x_grid = np.linspace(-12, 12, 200)
y_grid = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = ravine_loss(X, Y)

plt.figure(figsize=(12, 6))
cp = plt.contour(X, Y, Z, levels=np.logspace(-1, 2, 20), cmap="Blues_r")

plt.plot(
    traj_vanilla[:, 0],
    traj_vanilla[:, 1],
    "r-o",
    label="Vanilla GD (Violent y-oscillations, slow x-progress)",
    alpha=0.7,
)
plt.plot(
    traj_momentum[:, 0],
    traj_momentum[:, 1],
    "g-s",
    label="Momentum GD (Accelerates along x, damps y)",
    linewidth=2.5,
)
plt.plot(start_pt[0], start_pt[1], "k*", markersize=12, label="Start")
plt.plot(0, 0, "rX", markersize=10, label="Global Minimum")

plt.title("Vanilla GD vs Momentum GD on Ill-Conditioned Ravine (f = 0.05x^2 + y^2)", fontsize=13)
plt.xlabel("x (Flat direction)")
plt.ylabel("y (Steep direction)")
plt.legend(loc="upper right")
plt.grid(True, linestyle=":", alpha=0.5)
plt.show()

## 2. Navigating Non-Convex Landscapes and Saddle Points

Consider the non-convex double-well potential:

$$f(x, y) = x^4 - 2x^2 + y^2$$

Stationary points ($\nabla f = [4x^3 - 4x, 2y]^T = [0, 0]^T$):
1. **Saddle point** at $(0, 0)$ where the Hessian $H = \begin{bmatrix} -4 & 0 \\ 0 & 2 \end{bmatrix}$ has eigenvalues $\lambda_1 = -4$ (downward curve) and $\lambda_2 = +2$ (upward curve).
2. **Global minima** at $(+1, 0)$ and $(-1, 0)$ with $f = -1.0$.

If initialized near the saddle point $(0.05, 0.5)$, vanilla GD crawls slowly due to negligible gradient, whereas momentum builds velocity and quickly plunges toward the global minimum.


In [ ]:
def double_well(x, y):
    return x**4 - 2 * x**2 + y**2


def double_well_grad(x, y):
    return 4.0 * x**3 - 4.0 * x, 2.0 * y


def run_saddle_sim(init_pt, lr=0.08, steps=40):
    # Vanilla
    x, y = init_pt
    t_v = [(x, y)]
    for _ in range(steps):
        gx, gy = double_well_grad(x, y)
        x -= lr * gx
        y -= lr * gy
        t_v.append((x, y))

    # Momentum
    x, y = init_pt
    vx, vy = 0.0, 0.0
    t_m = [(x, y)]
    for _ in range(steps):
        gx, gy = double_well_grad(x, y)
        vx = 0.9 * vx + lr * gx
        vy = 0.9 * vy + lr * gy
        x -= vx
        y -= vy
        t_m.append((x, y))

    return np.array(t_v), np.array(t_m)


tv, tm = run_saddle_sim((0.02, 1.0), lr=0.05, steps=30)

loss_v = [double_well(pt[0], pt[1]) for pt in tv]
loss_m = [double_well(pt[0], pt[1]) for pt in tm]

plt.figure(figsize=(10, 4))
plt.plot(loss_v, "r--o", label="Vanilla GD Loss (Slow exit from saddle neighborhood)")
plt.plot(loss_m, "g-s", label="Momentum GD Loss (Rapid escape to global minimum -1.0)")
plt.xlabel("Iteration", fontsize=11)
plt.ylabel("Loss f(x, y)", fontsize=11)
plt.title("Escaping Saddle Points: Loss Trajectory Comparison", fontsize=13)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Final Vanilla Loss:  {loss_v[-1]:.4f}")
print(f"Final Momentum Loss: {loss_m[-1]:.4f}")
assert min(loss_m) < -0.99
assert min(loss_m[:16]) < min(loss_v[:16])